In [1]:
# ======================================================================
# 02_schema_audit.ipynb
# FinGap-Governance | Data-level audit of the raw AI-Hub datasets.
# ----------------------------------------------------------------------
# Tests the DATA-LEVEL claim at the precise granularity the review asks
# for. For each task we report, deterministically and platform-independently:
#   files / parsed  -- parse failures are counted, never silently dropped
#   determ_label    -- a determination/topic label is present
#   struct_link     -- an explicit STRUCTURED *paragraph-level* judgment-to-
#                      provision linkage field is present (the claim target)
#   clause_field    -- a STRUCTURED clause/standard-level reference field is
#                      present (accounting carries metadata.accounting_clause)
#   para_freetext   -- a paragraph-level operative reference appears only in
#                      the free-text answer
# This scoping matters: accounting DOES carry a structured clause field at
# chapter/standard granularity, so the honest claim is "no structured
# PARAGRAPH-LEVEL judgment-to-provision linkage field", not "no linkage
# information at all". Raw AI-Hub trees are expected under data/<task>/
# (not redistributed); if absent the notebook reports that and skips.
# ======================================================================
from fingap_core import *
import glob, json, re, csv

ADV = DATA / "advisory" / "labeled"
ANO = DATA / "anomaly"  / "raw"
ACC = DATA / "accounting" / "labeled"

# structured PARAGRAPH-LEVEL provision-linkage field names we look for (expect none)
LINK_KEYS = {"fcpa_article", "provision_id", "regulatory_phase", "suitability_phase",
             "appropriateness_phase", "explain_duty", "paragraph_id", "clause_paragraph",
             "조문링크", "근거조항", "근거문단", "operative_paragraph"}
# paragraph-level reference inside free text (항/호/조/N.N/para N/문단) -- 장(chapter) excluded
PARA_PAT = re.compile(r"(\d+\s*항|\d+\s*호|제\s*\d+\s*조|\d+\.\d+|문단|para(graph)?\s*\.?\s*\d+)")

def longpath(p):
    # Windows: bypass the 260-char MAX_PATH limit with the \\?\ extended-length
    # prefix (deep project folders + long Korean filenames exceed it otherwise).
    if os.name == "nt":
        ap = os.path.abspath(p)
        return ap if ap.startswith("\\\\?\\") else "\\\\?\\" + ap
    return p

def load_json_any(fp):
    last = None
    for enc in ("utf-8", "utf-8-sig", "cp949", "euc-kr"):
        try:
            with open(longpath(fp), encoding=enc) as f:
                return json.load(f), None
        except Exception as e:
            last = e
    return None, last

def flat_keys(o, acc):
    if isinstance(o, dict):
        for k, v in o.items():
            acc.add(k); flat_keys(v, acc)
    elif isinstance(o, list):
        for it in o: flat_keys(it, acc)

rows = []  # task, files, parsed, determ, struct_link, clause_field, para_freetext

# --- advisory (call-center consulting QA JSON) ------------------------
adv = sorted(glob.glob(str(ADV / "*.json")))
p = det = link = 0
for fp in adv:
    d, err = load_json_any(fp)
    if d is None: continue
    p += 1
    ks = set(); flat_keys(d, ks)
    topic = (d.get("consulting", {}) or {}).get("consulting_topic")
    det  += bool(topic or ({"qa_topic", "task_category"} & ks))
    link += bool(ks & LINK_KEYS)
rows.append(["advisory", len(adv), p, det, link, "n/a", "n/a"])

# --- anomaly (transaction CSV: inspect header columns) ----------------
ano = sorted(glob.glob(str(ANO / "*.csv")))
p = det = link = 0
for fp in ano:
    cols = []
    for enc in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
        try:
            with open(longpath(fp), encoding=enc) as f:
                cols = next(csv.reader(f), []); break
        except Exception:
            continue
    p += 1
    det  += bool({"이상거래유형", "이상거래여부", "이상거래설명"} & set(cols))
    link += bool({"조문", "근거조문", "STR", "CTR", "provision_id"} & set(cols))
rows.append(["anomaly", len(ano), p, det, link, "n/a", "n/a"])

# --- accounting (standards QA JSON) -----------------------------------
acc = sorted(glob.glob(str(ACC / "**" / "*.json"), recursive=True))
p = det = link = clause = para = 0
fail = 0
for fp in acc:
    d, err = load_json_any(fp)
    if d is None: fail += 1; continue
    p += 1
    ks = set(); flat_keys(d, ks)
    md = d.get("metadata", {}) or {}
    turns = (d.get("qa_content", {}) or {}).get("turns", []) or []
    answer = " ".join((t.get("answer") or "") for t in turns)
    if not answer:                                   # legacy 'output' field fallback
        answer = " ".join(str(x.get("output", "")) for x in (d.get("qa_data", []) or []))
    det    += bool(answer.strip())
    clause += bool((md.get("accounting_clause") or "").strip())   # structured clause field
    link   += bool(ks & LINK_KEYS)                                # structured para-level link
    para   += bool(PARA_PAT.search(answer))                       # para ref only in free text
rows.append(["accounting", len(acc), p, det, link, clause, para])

hdr = ["task", "files", "parsed", "determ", "struct_link", "clause_field", "para_freetext"]
w = [12, 7, 7, 8, 12, 13, 14]
print("".join(h.rjust(x) for h, x in zip(hdr, w)))
for r in rows:
    print("".join(str(v).rjust(x) for v, x in zip(r, w)))

out = TABLES / "schema_audit.csv"
with open(out, "w", newline="", encoding="utf-8-sig") as f:
    cw = csv.writer(f); cw.writerow(hdr); cw.writerows(rows)
print(f"\nsaved -> {out.name}")

if not (adv or ano or acc):
    print("\n[!] No raw AI-Hub data found under data/. Place the labelled trees there "
          "(see README) and re-run. The experiment notebooks 03-05 do NOT need this.")
else:
    acc_row = rows[-1]
    print("\nClaims to report (scoped to what the schemas actually contain):")
    print(" - advisory / anomaly: a determination/topic label is present, and there is NO "
          "structured judgment-to-provision linkage field.")
    print(f" - accounting: a STRUCTURED clause/standard-level field (metadata.accounting_clause) "
          f"is present for {acc_row[5]}/{acc_row[2]} parsed files, but there is NO structured "
          f"PARAGRAPH-LEVEL judgment-to-provision linkage field ({acc_row[4]}); the operative "
          f"paragraph-level basis appears only in free-text answers ({acc_row[6]}/{acc_row[2]}).")
    print("   => report as 'no structured paragraph-level linkage field', not 'no linkage information'.")


        task  files parsed  determ struct_link clause_field para_freetext
    advisory      5      5       5           0          n/a           n/a
     anomaly      1      1       1           0          n/a           n/a
  accounting    775    775     775           0          775           493

saved -> schema_audit.csv

Claims to report (scoped to what the schemas actually contain):
 - advisory / anomaly: a determination/topic label is present, and there is NO structured judgment-to-provision linkage field.
 - accounting: a STRUCTURED clause/standard-level field (metadata.accounting_clause) is present for 775/775 parsed files, but there is NO structured PARAGRAPH-LEVEL judgment-to-provision linkage field (0); the operative paragraph-level basis appears only in free-text answers (493/775).
   => report as 'no structured paragraph-level linkage field', not 'no linkage information'.
